# Intraday Momentum on SPY – Backtest Notebook

Replication of Zarattini, Aziz & Barbon (2024) and evaluation of our own extensions.

All logic lives in the package `src/intraday_momentum/` and is **only imported here**. The notebook orchestrates, displays and discusses. Reading guide:

| Topic | Document |
|---|---|
| Math of the strategy | [`docs/strategy.md`](../docs/strategy.md) |
| Own extensions (design, hypotheses) | [`docs/extensions.md`](../docs/extensions.md) |
| Report, incl. cross-check with the authors' code | [`docs/report.md`](../docs/report.md) |

**Contents**
1. Setup
2. Data
3. The Noise Area on an example day
4. Paper replication: train vs. test
5. Cost sensitivity
6. Robustness (train period only)
7. Trade-level diagnostics
8. Own strategy (Idea 1: spike filter, Idea 2: meta-labeling)
9. Conclusions

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:
    import intraday_momentum  # installed via `pip install -e .`
except ModuleNotFoundError:
    sys.path.insert(0, str(ROOT / "src"))  # fallback: import directly from src/

%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [ ]:
from intraday_momentum import run_backtest, summarize
from intraday_momentum.evaluation import (COST_SCENARIOS, PAPER_VARIANTS, cost_sensitivity, evaluate, format_table,
                                          load_project_data, make_periods, metric_bar_table, parameter_grid,
                                          summary_view)
from intraday_momentum.diagnostics import false_breakout_stats, pnl_by_entry_time, pnl_by_side, yearly_returns
from intraday_momentum.plotting import (plot_cost_sensitivity, plot_equity_curves, plot_metric_bars,
                                        plot_noise_area_day)

**Parameters.** The split date is fixed *before* looking at any test results. `USE_SYNTHETIC = True` runs the notebook without downloaded data.

In [ ]:
SPLIT = "2022-01-01"      # first day of the test period
USE_SYNTHETIC = False     # True -> random-walk data (no API key needed)
SAVE_FIGURES = True       # also write figures to results/

## 2. Data

SPY 1-minute bars (Alpaca SIP, unadjusted, regular session) plus dividends, reshaped into *day × minute* matrices (`data.build_day_data`). The benchmark is SPY buy & hold from adjusted daily closes. If `data/raw` is empty, synthetic data is used automatically.

In [ ]:
data, bench = load_project_data(ROOT, synthetic=USE_SYNTHETIC)
if USE_SYNTHETIC or len(data) == 600:
    SPLIT = "2022-06-01"  # synthetic sample is shorter
periods = make_periods(data, SPLIT)
OUT = ROOT / "results" / ("synthetic" if len(data) == 600 else "")
OUT.mkdir(parents=True, exist_ok=True)
save = (lambda name: OUT / name) if SAVE_FIGURES else (lambda name: None)

print(f"{len(data)} trading days: {data.dates[0].date()} -> {data.dates[-1].date()}")
print(f"half days (early close): {(data.n_bars < 390).sum()}")
pd.DataFrame(periods, index=["start", "end"]).T

## 3. The Noise Area on an example day

Yellow: Noise Area $[\mathrm{LB}_{t,k}, \mathrm{UB}_{t,k}]$. Red dotted line: VWAP. Grey lines: decision times (every 30 minutes). Triangles mark entries and crosses mark exits of the band/VWAP variant.

In [ ]:
ext1 = run_backtest(data, PAPER_VARIANTS["Ext. 1: + band/VWAP stop"], COST_SCENARIOS["paper"], start=periods["Train"][0])
example_day = str(ext1.returns.idxmax().date())   # best day; try any other date, e.g. "2022-01-20"
plot_noise_area_day(data, example_day, save("noise_area_example.png"), trades=ext1.trades);

## 4. Paper replication: train vs. test

Three variants (base, + band/VWAP stop, + vol targeting) × two cost scenarios × two periods. Each period starts with \$100k. All parameters are the paper's, so **nothing is fitted on the test period**.

In [ ]:
summary, full = evaluate(data, PAPER_VARIANTS, COST_SCENARIOS, periods, bench)
summary.to_csv(OUT / "summary.csv", index=False)
format_table(summary_view(summary, "paper"))

In [ ]:
format_table(summary_view(summary, "conservative"))

In [ ]:
order = list(PAPER_VARIANTS) + ["SPY buy & hold"]
plot_metric_bars(metric_bar_table(summary, "paper", order), save("metrics_train_vs_test.png"));

In [ ]:
curves = {name: full[("paper", name)].returns for name in PAPER_VARIANTS}
curves["SPY buy & hold"] = bench.loc[periods["Train"][0]:]
plot_equity_curves(curves, SPLIT, save("equity_curves.png"), "Intraday momentum on SPY vs buy & hold (paper costs)");

**Discussion** *(fill in after the run)*
- Does the ranking of the variants (base < VWAP stop < vol targeting) from the paper hold out of sample?
- Sharpe in train vs. test: how large is the drop?
- How much of the return of Ext. 2 is leverage? Compare Sharpe, not total return.

## 5. Cost sensitivity (test period)

Sharpe ratio as a function of total cost per share (commission \$0.0035 + slippage). Where is break-even?

In [ ]:
sens = cost_sensitivity(data, PAPER_VARIANTS, periods["Test"])
sens.to_csv(OUT / "cost_sensitivity.csv")
plot_cost_sensitivity(sens, save("cost_sensitivity.png"))
sens.round(2)

## 6. Robustness: lookback × volatility multiplier (train period only)

A flat plateau is a good sign, a single sharp peak is a warning sign of overfitting. The **test period is not used here**. The best VM from this grid is also the benchmark for Idea 1 (see `extensions.md` §0.1).

In [ ]:
grid = parameter_grid(data, PAPER_VARIANTS["Ext. 2: + vol targeting"], COST_SCENARIOS["paper"], periods["Train"])
grid.to_csv(OUT / "robustness_train_sharpe.csv")
grid.style.format("{:.2f}").background_gradient(cmap="RdYlGn", axis=None)

## 7. Trade-level diagnostics (train period)

Where does the P&L come from? Only the **training period** is used here, because these analyses motivate our own extensions.

In [ ]:
train_s, train_e = periods["Train"]
pnl_by_side(ext1, train_s, train_e).round(3)

In [ ]:
# cf. paper FAQ Q18: do trends pause at lunch time?
by_time = pnl_by_entry_time(ext1, train_s, train_e)
ax = by_time["avg_bps"].plot.bar(figsize=(9, 3.5), color="#1f3b73", title="Average gross return per trade by entry time (bp)")
ax.axhline(0, color="k", lw=0.8);
by_time.round(3).T

In [ ]:
# Motivation for Idea 1: how many entries are stopped out again at the very next decision time?
false_breakout_stats(ext1, train_s, train_e).round(3)

In [ ]:
yearly_returns({name: full[("paper", name)].returns for name in PAPER_VARIANTS} | {"SPY": bench.dropna()}).style.format("{:.1%}")

## 8. Own strategy: ML long/short/flat model

Design and reasoning: [`docs/extensions.md`](../docs/extensions.md). In short:

- **Same decision grid as the paper** (every 30 min from 10:00, flat at the close) and the same execution, costs and sizing. Only the *decision rule* changes.
- **Model:** logistic regression (L2, standardised features) for $P(\text{return from now to the close} > 0)$.
- **Position:** $+1$ if $p > 0.5 + m$, $-1$ if $p < 0.5 - m$, else flat. Re-evaluated every 30 minutes, so a model turning neutral acts as the stop.
- **Features:** 7 features inspired by the paper, all known at the decision time (unit-tested for look-ahead).
- **Selection:** feature set, $C$ and margin $m$ are chosen by expanding-window CV by year **on the training period only** (validation years 2018–2021). Then one final fit on 2016–2021 and one evaluation on the test period.

In [ ]:
from intraday_momentum.features import FEATURE_DESCRIPTIONS
from intraday_momentum.ml_strategy import ML_VARIANTS, ml_pipeline
from intraday_momentum.evaluation import SUBPERIODS, subperiod_table

pd.Series(FEATURE_DESCRIPTIONS, name="definition").to_frame()

In [ ]:
ml_run = ml_pipeline(data, periods, COST_SCENARIOS, bench)   # CV + final fit + evaluation (~10 s)
print("Selected:", ml_run.cfg)
ml_run.feature_set_comparison.round(3)

**Cross-validation on the training period** (mean over the validation years 2018–2021). The margin controls how often the model trades and therefore the costs.

In [ ]:
ml_run.cv_table.round(3)

In [ ]:
# Per validation year for the selected parameters: is the result stable over time?
sel = ml_run.cv[(ml_run.cv.C == ml_run.cfg.C) & (ml_run.cv.margin == ml_run.cfg.margin)]
sel.set_index("year")[["sharpe", "ann_return", "auc", "trades_per_day"]].round(3)

**What did the model learn?** Coefficients on standardised features (sign = direction of the effect on $P(\text{up})$). AUC 0.5 = no skill.

In [ ]:
ax = ml_run.coefficients.sort_values().plot.barh(figsize=(7, 3.5), color="#1f3b73", title="Logistic regression coefficients (standardised)")
ax.axvline(0, color="k", lw=0.8)
display(ml_run.coefficients.round(3).to_frame("coef"))
ml_run.auc.round(3).to_frame("AUC")

### 8.1 Comparison with the paper rule (same split, costs and metrics)

In [ ]:
paper_sel = {k: PAPER_VARIANTS[k] for k in ["Ext. 1: + band/VWAP stop", "Ext. 2: + vol targeting"]}
cmp_summary = pd.concat([summary[summary.strategy.isin(list(paper_sel) + ["SPY buy & hold"])], ml_run.summary],
                        ignore_index=True)
format_table(summary_view(cmp_summary, "paper"))

In [ ]:
format_table(summary_view(cmp_summary, "conservative"))

In [ ]:
ml_order = list(paper_sel) + list(ML_VARIANTS) + ["SPY buy & hold"]
plot_metric_bars(metric_bar_table(cmp_summary, "paper", ml_order), save("ml_metrics_train_vs_test.png"));

In [ ]:
test_start = periods["Test"][0]
all_res = {**full, **ml_run.full}
ml_curves = {n: all_res[("paper", n)].returns.loc[test_start:] for n in ml_order[:-1]}
ml_curves["SPY buy & hold"] = bench.loc[test_start:]
plot_equity_curves(ml_curves, None, save("ml_equity_test.png"), "Test period: ML strategy vs paper rule vs SPY");

### 8.2 Before vs. after the paper's publication (May 2024), and diversification

In [ ]:
sub_returns = {"Ext. 1: + band/VWAP stop": full[("paper", "Ext. 1: + band/VWAP stop")].returns,
               "ML: logistic (1x)": ml_run.full[("paper", "ML: logistic (1x)")].returns}
display(format_table(subperiod_table(sub_returns, SUBPERIODS, bench)))
corr = pd.concat(sub_returns, axis=1).loc[test_start:].corr().iloc[0, 1]
print(f"Correlation of daily returns, ML vs Ext. 1 (test period): {corr:.2f}")

**Discussion** *(see `docs/report.md` §8 for the written version)*
- Does the ML model beat the rule? Compare the test Sharpe ratios after costs.
- Is the edge real? AUC on the test period, sign and stability of the coefficients.
- Train vs. test: the in-sample train result is weak, so the model is not simply overfitted.
- Diversification: the ML returns are almost uncorrelated with the rule.

## 9. Conclusions

*(fill in after the run)*
- Replication: …
- Out-of-sample: …
- Costs: …
- Own strategy: …
- Further improvements: multi-asset, regime-aware sizing (see `extensions.md` §7)